# 02. Model Training and Evaluation Pipeline
We fit, tune, and compare Bernoulli Naive Bayes, Multinomial Naive Bayes, and Linear Support Vector Classifiers (LinearSVC).

### Step 1: Load Split Datasets
We read `/content/train_split.csv`, `/content/val_split.csv`, and `/content/test_split.csv` splits created during preprocessing.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

train_df = pd.read_csv('/content/train_split.csv')
val_df = pd.read_csv('/content/val_split.csv')
test_df = pd.read_csv('/content/test_split.csv')

text_col = 'tweet_variant_b'
target_col = 'Party'

train_df = train_df.dropna(subset=[text_col, target_col])
val_df = val_df.dropna(subset=[text_col, target_col])
test_df = test_df.dropna(subset=[text_col, target_col])

print(f"Loaded Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")

### Step 2: Establish Evaluation Strategy
We map Party categories (Democrat -> 0, Republican -> 1) and create a unified evaluation function. Vectorizers are fitted exclusively on the Training set.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import BernoulliNB, MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import joblib

y_train = (train_df[target_col] == 'Republican').astype(int).values
y_val = (val_df[target_col] == 'Republican').astype(int).values
y_test = (test_df[target_col] == 'Republican').astype(int).values

X_train = train_df[text_col].values
X_val = val_df[text_col].values
X_test = test_df[text_col].values

performance_metrics = {}
trained_pipelines = {}

def evaluate_pipeline(name, pipeline):
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')
    
    performance_metrics[name] = {
        'Accuracy': acc,
        'Macro Precision': prec,
        'Macro Recall': rec,
        'Macro F1': f1,
        'Predictions': y_pred
    }
    trained_pipelines[name] = pipeline
    print(f"--- {name} Results ---")
    print(classification_report(y_test, y_pred, target_names=['Democrat', 'Republican']))
    return pipeline

### Step 3: Model Execution and Tuning
We initialize and evaluate the three pipelines sequentially.

In [ ]:
# Bernoulli Naive Bayes (Binary features)
pipe_bnb = Pipeline([
    ('vectorizer', CountVectorizer(binary=True)),
    ('classifier', BernoulliNB())
])
evaluate_pipeline('Bernoulli Naive Bayes', pipe_bnb)

# Multinomial Naive Bayes (Frequency counts)
pipe_mnb = Pipeline([
    ('vectorizer', CountVectorizer(binary=False)),
    ('classifier', MultinomialNB())
])
evaluate_pipeline('Multinomial Naive Bayes', pipe_mnb)

# Linear SVM (TF-IDF features)
pipe_svc = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', LinearSVC(random_state=42, dual=False))
])
evaluate_pipeline('Linear SVM (LinearSVC)', pipe_svc)

### Step 4: Comparison and Artifact Preservation
We present the final comparison table and persist the best performing pipeline model state on disk.

In [ ]:
comparison_list = []
for name, m in performance_metrics.items():
    comparison_list.append({
        'Model': name,
        'Accuracy': f"{m['Accuracy']:.4f}",
        'Macro Precision': f"{m['Macro Precision']:.4f}",
        'Macro Recall': f"{m['Macro Recall']:.4f}",
        'Macro F1': f"{m['Macro F1']:.4f}"
    })

df_results = pd.DataFrame(comparison_list)
display(df_results)

# Save the highest scoring pipeline
best_model_name = df_results.sort_values(by='Macro F1', ascending=False).iloc[0]['Model']
best_pipeline = trained_pipelines[best_model_name]
joblib.dump(best_pipeline, '/content/best_model_pipeline.joblib')
print(f"Successfully serialized best model [{best_model_name}] pipeline!")